# Notebook 09 — National Model: Full Results & Visualisations

Mirrors Notebook 04 (Nyungwe only) but for the **national model — all 5 Rwanda provinces, 23,319 pixels**.  
Adds: per-fold spatial CV, province leave-one-out, 17-feature importance, P-R curve, Hansen external validation.

Run **after** Notebook 06 (`06_National_Extension.ipynb`).  
Most charts load from pre-saved JSON — the confusion matrix and P-R curve cells refit a 200-tree RF (≈60s).

| Chart | What it shows |
|---|---|
| 1 | Dataset: province × class distribution |
| 2 | RQ1 — F1 by experiment (Nyungwe vs National) |
| 3 | Feature importance — all 17 features, coloured by source |
| 4 | Feature importance by source group (radar vs optical vs terrain) |
| 5 | Spatial CV — honest vs optimistic F1 + per-fold scatter |
| 6 | Leave-one-province-out generalisation |
| 7 | Confusion matrix (national model D, 5-fold CV) |
| 8 | Precision-Recall curve |
| 9 | RQ2 — Recall by real clearing-patch size (Hansen GFC) |
| 10 | External validation: TreeSight vs Hansen sector-level deforestation % |

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                              confusion_matrix, precision_recall_curve, average_precision_score)
import warnings
warnings.filterwarnings('ignore')

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

METRICS = 'results/metrics'
GREEN   = '#1B5E20'
AMBER   = '#E65100'
BLUE    = '#1565C0'
GREY    = '#757575'
SOURCE_COLORS = {'optical': BLUE, 'radar': AMBER, 'terrain': '#2E7D32'}

OPTICAL  = ['EVI_train','NBR_train','NDVI_change','NDVI_test','NDVI_train','NIR_train','RED_train','SWIR_test','SWIR_train']
TERRAIN  = ['aspect','elevation','slope']
RADAR    = ['VH_VV_ratio','VH_test','VH_train','VV_test','VV_train']
FEAT_D   = OPTICAL + TERRAIN + RADAR

def source_of(feat):
    if feat in TERRAIN: return 'terrain'
    if feat in RADAR:   return 'radar'
    return 'optical'

# Load data
raw   = pd.read_csv('data/raw/training_data_national.csv')
nat   = raw.drop(columns=['system:index','.geo'], errors='ignore')
X_D   = nat[FEAT_D].values
y     = nat['label'].values.astype(int)

# Load saved results
nc    = json.load(open(f'{METRICS}/national_comparison.json'))
na    = json.load(open(f'{METRICS}/national_rq_analysis.json'))
rq2   = json.load(open(f'{METRICS}/rq2_patchsize.json'))
hb    = json.load(open(f'{METRICS}/hansen_benchmark.json'))
scv   = json.load(open(f'{METRICS}/spatial_cv_results.json'))

print(f'National dataset : {len(nat):,} pixels across {nat["province"].nunique()} provinces')
print(f'Deforested (1)   : {(y==1).sum():,}  |  Intact (0): {(y==0).sum():,}')
print(f'17 features  → D : {len(FEAT_D)} cols confirmed')
print('Results loaded   :', list(nc.keys()))

## Chart 1 — Dataset: province breakdown & class balance

In [ ]:
prov_counts = nat.groupby(['province','label']).size().unstack(fill_value=0).rename(columns={0:'Intact',1:'Deforested'})
prov_counts.index = [p.split('/')[0] for p in prov_counts.index]  # short names

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Stacked bar — pixels per province
prov_counts[['Intact','Deforested']].plot(kind='bar', stacked=True,
    color=['#90A4AE','#1B5E20'], ax=axes[0], width=0.65, edgecolor='white')
axes[0].set_title('Pixels per province', fontsize=12, fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Pixels')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(['Intact forest','Deforested'], frameon=False)
axes[0].grid(axis='y', alpha=0.3)
for bar in axes[0].patches:
    h = bar.get_height()
    if h > 300:
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_y()+h/2,
                     f'{int(h):,}', ha='center', va='center', fontsize=7.5, color='white', fontweight='bold')

# Class balance pie
total_defo = int((y==1).sum())
total_int  = int((y==0).sum())
axes[1].pie([total_int, total_defo],
            labels=[f'Intact\n{total_int:,}', f'Deforested\n{total_defo:,}'],
            colors=['#90A4AE','#1B5E20'], autopct='%1.1f%%',
            startangle=90, textprops={'fontsize':11})
axes[1].set_title(f'Class balance\n(n = {len(nat):,} total)', fontsize=12, fontweight='bold')

plt.suptitle('National dataset — 5 provinces', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{METRICS}/nat_chart1_dataset.png', dpi=150, bbox_inches='tight')
plt.show()

## Chart 2 — RQ1: F1 by experiment (Nyungwe vs National)
All 4 feature-set experiments under identical 5-fold stratified CV, 200 trees.

In [ ]:
exps = nc['four_experiments_5foldcv_200trees']
exp_names = [e['experiment'] for e in exps]
nyu_f1    = [e['nyungwe_f1']  for e in exps]
nat_f1    = [e['national_f1'] for e in exps]
nat_std   = [e['national_std'] for e in exps]
nyu_std   = [e['nyungwe_std']  for e in exps]

x = np.arange(len(exp_names))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - w/2, nyu_f1, w, yerr=nyu_std, capsize=4,
               color='#90A4AE', edgecolor='white', label='Nyungwe (10k px)')
bars2 = ax.bar(x + w/2, nat_f1, w, yerr=nat_std, capsize=4,
               color=GREEN, edgecolor='white', label='National (23k px)')

for bar, val, std in zip(bars1, nyu_f1, nyu_std):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+std+0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, color='#455A64')
for bar, val, std in zip(bars2, nat_f1, nat_std):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+std+0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9.5, color=GREEN, fontweight='bold')

ax.axhline(0.71, color='red', linewidth=1.2, linestyle='--', alpha=0.7)
ax.text(3.65, 0.713, 'baseline F1 = 0.71', fontsize=8.5, color='red')

short_names = ['A — Optical\n(9 feat)', 'B — Optical+\nTerrain (12)', 'C — Optical+\nRadar (14)', 'D — All\ncombined (17)']
ax.set_xticks(x)
ax.set_xticklabels(short_names, fontsize=10)
ax.set_ylabel('F1 Score (5-fold CV)', fontsize=11)
ax.set_title('RQ1 — Effect of adding radar and terrain on detection F1', fontsize=13, fontweight='bold')
ax.set_ylim(0.60, 0.89)
ax.legend(frameon=False, fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)

# Annotate delta for national
for i, e in enumerate(exps):
    if i > 0:
        delta = e['delta']
        ax.annotate('', xy=(x[i]+w/2, nat_f1[i]), xytext=(x[i-1]+w/2, nat_f1[i-1]),
                    arrowprops=dict(arrowstyle='->', color=GREEN, lw=1.2))

plt.tight_layout()
plt.savefig(f'{METRICS}/nat_chart2_rq1_experiments.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nGains for national model:')
for i,e in enumerate(exps):
    if i > 0:
        print(f'  {exps[i-1]["experiment"].split(" ")[0]} → {e["experiment"].split(" ")[0]}: +{e["delta"]:.3f}')

## Chart 3 — Feature importance: all 17 features, coloured by source
Mean decrease in impurity from the national model D (800 trees, depth 25).

In [ ]:
fi = pd.DataFrame(na['feature_importance']).sort_values('importance', ascending=True)
colors = [SOURCE_COLORS[source_of(f)] for f in fi['feature']]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(fi['feature'], fi['importance'], color=colors, edgecolor='white', height=0.75)

for bar, val in zip(bars, fi['importance']):
    ax.text(val + 0.001, bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8.5)

legend_patches = [mpatches.Patch(color=SOURCE_COLORS[s], label=s.capitalize()) for s in ['optical','radar','terrain']]
ax.legend(handles=legend_patches, loc='lower right', frameon=False, fontsize=10)

ax.set_xlabel('Mean decrease in impurity (normalised)', fontsize=10)
ax.set_title('Feature importance — National model D (all 17 features)', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
ax.spines[['top','right']].set_visible(False)

# Source totals annotation
by_src = na['by_source']
summary = f"Optical {by_src['optical']:.1%}  |  Radar {by_src['radar']:.1%}  |  Terrain {by_src['terrain']:.1%}"
ax.set_xlabel(f'Mean decrease in impurity\n{summary}', fontsize=10)

plt.tight_layout()
plt.savefig(f'{METRICS}/nat_chart3_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Chart 4 — Feature importance by source group

In [ ]:
# Hardcoded Nyungwe by-source (from spatial CV model, approximate)
NYU_BY_SRC = {'optical': 0.537, 'radar': 0.218, 'terrain': 0.245}
NAT_BY_SRC = na['by_source']

sources = ['optical', 'radar', 'terrain']
x = np.arange(len(sources))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 4.5))
b1 = ax.bar(x - w/2, [NYU_BY_SRC[s] for s in sources], w, color='#90A4AE', label='Nyungwe')
b2 = ax.bar(x + w/2, [NAT_BY_SRC[s] for s in sources], w,
            color=[SOURCE_COLORS[s] for s in sources], label='National')

for bar in b1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{bar.get_height():.1%}', ha='center', fontsize=9, color='#546E7A')
for bar in b2:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{bar.get_height():.1%}', ha='center', fontsize=9.5, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([s.capitalize() for s in sources], fontsize=11)
ax.set_ylabel('Share of total feature importance', fontsize=10)
ax.set_title('Feature importance by source — Nyungwe vs National', fontsize=12, fontweight='bold')
ax.set_ylim(0, 0.65)
ax.legend(frameon=False, fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(f'{METRICS}/nat_chart4_importance_by_source.png', dpi=150, bbox_inches='tight')
plt.show()

print('National: radar contributes', f"{NAT_BY_SRC['radar']:.1%}",
      '— 2nd-largest source despite optical dominance.')

## Chart 5 — Spatial CV: honest vs random F1, per-fold distribution
**Cite the spatial-block F1 in the abstract/conclusion** (D-014).  
Random CV overfits because nearby pixels share the same spatial autocorrelation.

In [ ]:
spatial_info = nc['national_spatial_block_cv']
per_fold     = spatial_info['per_fold']
spatial_f1   = spatial_info['f1']
spatial_std  = spatial_info['f1_std']

# Nyungwe comparison from spatial_cv_results.json
nyu_random  = scv['random_cv']['f1']
nyu_spatial = scv['spatial_block_cv']['f1']

nat_random  = nc['four_experiments_5foldcv_200trees'][-1]['national_f1']  # D experiment 5-fold

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: Nyungwe vs National bar comparison
labels = ['Nyungwe\nRandom CV', 'Nyungwe\nSpatial CV', 'National\nRandom CV', 'National\nSpatial CV']
values = [nyu_random, nyu_spatial, nat_random, spatial_f1]
colors = ['#90A4AE','#E57373','#90CAF9', GREEN]
bars = axes[0].bar(labels, values, color=colors, width=0.6, edgecolor='white')
for bar, val in zip(bars, values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.004,
                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_ylim(0.65, 0.87)
axes[0].axhline(0.71, color='red', linestyle='--', linewidth=1, alpha=0.6)
axes[0].set_ylabel('F1 Score', fontsize=10)
axes[0].set_title('Random vs Spatial CV — both regions', fontsize=11, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
axes[0].spines[['top','right']].set_visible(False)
axes[0].annotate('← optimistic (spatial leakage)',
                 xy=(0, nyu_random), xytext=(0.3, nyu_random+0.02),
                 fontsize=8, color='#546E7A', arrowprops=dict(arrowstyle='->', color='#546E7A'))

# Right: per-fold spatial CV for national
fold_labels = [f'Fold {i+1}' for i in range(len(per_fold))]
fold_colors = [GREEN if v >= 0.75 else AMBER for v in per_fold]
axes[1].bar(fold_labels, per_fold, color=fold_colors, width=0.6, edgecolor='white')
axes[1].axhline(spatial_f1, color='black', linestyle='--', linewidth=1.5,
                label=f'Mean = {spatial_f1:.3f} ± {spatial_std:.3f}')
axes[1].axhline(0.75, color='red', linestyle=':', linewidth=1, alpha=0.5)
for i, (label, val) in enumerate(zip(fold_labels, per_fold)):
    axes[1].text(i, val+0.005, f'{val:.3f}', ha='center', fontsize=9.5, fontweight='bold')
axes[1].set_ylim(0.55, 0.90)
axes[1].set_ylabel('F1 Score', fontsize=10)
axes[1].set_title('National — per-fold spatial block CV\n(10 KMeans blocks, GroupKFold-5, 800 trees)',
                  fontsize=11, fontweight='bold')
axes[1].legend(frameon=False, fontsize=10)
axes[1].grid(axis='y', alpha=0.3)
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(f'{METRICS}/nat_chart5_spatial_cv.png', dpi=150, bbox_inches='tight')
plt.show()

drop = nat_random - spatial_f1
print(f'Random CV F1  : {nat_random:.3f}  (optimistic — spatial leakage)')
print(f'Spatial CV F1 : {spatial_f1:.3f} ± {spatial_std:.3f}  ← cite this in abstract')
print(f'Drop          : -{drop:.3f} ({drop/nat_random:.1%})')

## Chart 6 — Leave-one-province-out generalisation
Each province is held out as an unseen test set while training on the other four.  
This tests geographic transferability — can the model generalise to new regions?

In [ ]:
lopo = nc['leave_one_province_out']
prov_names = list(lopo.keys())
prov_short = [p.split('/')[0] for p in prov_names]  # English part only
prov_f1    = [lopo[p]['f1']    for p in prov_names]
prov_rec   = [lopo[p]['recall'] for p in prov_names]
prov_n     = [lopo[p]['n_test'] for p in prov_names]

x = np.arange(len(prov_short))
w = 0.40

fig, ax = plt.subplots(figsize=(10, 5))
bar_f1  = ax.bar(x - w/2, prov_f1,  w, color=[GREEN if v>=0.75 else AMBER for v in prov_f1],
                 label='F1', edgecolor='white')
bar_rec = ax.bar(x + w/2, prov_rec, w, color=['#66BB6A' if v>=0.75 else '#FFA726' for v in prov_rec],
                 label='Recall', edgecolor='white', alpha=0.85)

for bar, val in zip(bar_f1, prov_f1):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')
for bar, val in zip(bar_rec, prov_rec):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{val:.3f}', ha='center', fontsize=9)

# Annotate n_test
for i, n in enumerate(prov_n):
    ax.text(i, 0.62, f'n={n:,}', ha='center', fontsize=8, color='#546E7A')

ax.axhline(0.75, color='red', linestyle='--', linewidth=1.2, alpha=0.6)
ax.text(len(prov_short)-0.45, 0.753, 'F1 = 0.75 target', fontsize=8.5, color='red')

ax.set_xticks(x)
ax.set_xticklabels(prov_short, fontsize=10, rotation=15, ha='right')
ax.set_ylim(0.60, 0.92)
ax.set_ylabel('Score', fontsize=10)
ax.set_title('Leave-one-province-out — F1 and Recall per held-out province', fontsize=12, fontweight='bold')
ax.legend(frameon=False, fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(f'{METRICS}/nat_chart6_province_lopo.png', dpi=150, bbox_inches='tight')
plt.show()

weak = min(prov_names, key=lambda p: lopo[p]['f1'])
print(f'Weakest province: {weak.split("/")[0]} — F1 {lopo[weak]["f1"]:.3f}')
print('Interpretation: West has fewest training pixels from other provinces → generalises worst.')

## Chart 7 — Confusion matrix (national model D, 5-fold CV)
Out-of-fold predictions from `cross_val_predict` — no data leakage. May take ~60s.

In [ ]:
rf200 = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
cv    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('Running 5-fold cross_val_predict (200 trees)…')
y_pred = cross_val_predict(rf200, X_D, y, cv=cv, n_jobs=-1)
y_prob = cross_val_predict(rf200, X_D, y, cv=cv, method='predict_proba', n_jobs=-1)[:,1]

f1  = f1_score(y, y_pred)
prec = precision_score(y, y_pred)
rec  = recall_score(y, y_pred)
auc  = roc_auc_score(y, y_prob)
print(f'F1={f1:.3f}  Precision={prec:.3f}  Recall={rec:.3f}  AUC={auc:.3f}')

cm = confusion_matrix(y, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

labels_str = [['True Negative\n(Correctly intact)', 'False Positive\n(False alarm)'],
               ['False Negative\n(Missed clearing)', 'True Positive\n(Correctly detected)']]

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(cm_pct, cmap='Greens', vmin=0, vmax=100)
plt.colorbar(im, ax=ax, label='% of actual class')

for i in range(2):
    for j in range(2):
        color = 'white' if cm_pct[i,j] > 55 else 'black'
        ax.text(j, i, f'{cm[i,j]:,}\n({cm_pct[i,j]:.1f}%)\n{labels_str[i][j]}',
                ha='center', va='center', fontsize=9.5, color=color)

ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Predicted\nIntact', 'Predicted\nDeforested'], fontsize=10)
ax.set_yticklabels(['Actual\nIntact', 'Actual\nDeforested'], fontsize=10)
ax.set_title(f'Confusion matrix — National model D (5-fold OOF)\nF1={f1:.3f}  Precision={prec:.3f}  Recall={rec:.3f}  AUC={auc:.3f}',
             fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{METRICS}/nat_chart7_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Chart 8 — Precision-Recall curve

In [ ]:
precision_pts, recall_pts, thresholds = precision_recall_curve(y, y_prob)
ap = average_precision_score(y, y_prob)

# Default threshold operating point (0.5)
default_prec = precision_score(y, y_pred)
default_rec  = recall_score(y, y_pred)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(recall_pts, precision_pts, color=GREEN, linewidth=2.5, label=f'PR curve (AP = {ap:.3f})')
ax.scatter([default_rec], [default_prec], color='red', zorder=5, s=80,
           label=f'Default threshold (0.5)\nP={default_prec:.3f}  R={default_rec:.3f}')

# Shade area under curve
ax.fill_between(recall_pts, precision_pts, alpha=0.08, color=GREEN)

# Reference: random classifier
baseline = (y==1).mean()
ax.axhline(baseline, color=GREY, linestyle='--', linewidth=1, label=f'Random baseline ({baseline:.2f})')

ax.set_xlabel('Recall (sensitivity)', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Precision-Recall curve — National model D (5-fold OOF)', fontsize=12, fontweight='bold')
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.05)
ax.legend(frameon=False, fontsize=10, loc='lower left')
ax.grid(alpha=0.3)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(f'{METRICS}/nat_chart8_pr_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Average Precision (area under PR): {ap:.3f}')
print('Move threshold left on the curve to boost recall (catch more clearings) at some precision cost.')

## Chart 9 — RQ2: Detection recall by real clearing-patch size (Hansen GFC)
Recall measured per connected-component patch-size bucket using out-of-fold predictions.  
Real clearing sizes from `connectedPixelCount` (Hansen 2020-2022 loss window).

In [ ]:
buckets = [b['bucket'].replace('\u2013','-') for b in rq2['by_patch_size']]
recalls  = [b['recall'] for b in rq2['by_patch_size']]
ns       = [b['n']      for b in rq2['by_patch_size']]

colors = ['#E65100' if r < 0.80 else '#1B5E20' for r in recalls]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(buckets)), recalls, color=colors, width=0.65, edgecolor='white')

for i, (bar, val, n) in enumerate(zip(bars, recalls, ns)):
    ax.text(i, val + 0.005, f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
    ax.text(i, 0.68, f'n={n:,}', ha='center', fontsize=8.5, color='#546E7A')

ax.axhline(0.75, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
ax.text(len(buckets)-0.45, 0.753, 'target = 0.75', fontsize=9, color='red')

ax.set_xticks(range(len(buckets)))
ax.set_xticklabels(buckets, fontsize=9.5)
ax.set_xlabel('Real clearing-patch size (Hansen connectedPixelCount)', fontsize=10)
ax.set_ylabel('Recall (share of clearings detected)', fontsize=10)
ax.set_title('RQ2 — Detection recall by patch size: recall rises monotonically with clearing area',
             fontsize=12, fontweight='bold')
ax.set_ylim(0.65, 0.93)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)

# Arrow showing trend
ax.annotate('', xy=(len(buckets)-1, recalls[-1]-0.01), xytext=(0, recalls[0]+0.01),
            arrowprops=dict(arrowstyle='->', color=GREEN, lw=1.5, connectionstyle='arc3,rad=0.1'))
ax.text(2.0, 0.865, 'improving →', fontsize=9, color=GREEN, style='italic')

plt.tight_layout()
plt.savefig(f'{METRICS}/nat_chart9_rq2_patchsize.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Smallest bucket (0.09-0.18 ha): recall = {recalls[0]:.3f}')
print(f'Largest bucket  (>1.8 ha)     : recall = {recalls[-1]:.3f}')
print(f'Overall recall across all patches: {rq2["overall_recall"]:.3f}')

## Chart 10 — External validation: TreeSight vs Hansen GFC at sector level
Independent check: do our sector-level deforestation predictions correlate with Hansen Global Forest Change?  
Each point = one of 365 Rwanda sectors. Both axes = % of sector pixels classified as deforested.

In [ ]:
sectors = pd.DataFrame(hb['per_sector'])

RISK_COLORS = {'HIGH':'#B71C1C','MEDIUM':'#E65100','LOW':'#1B5E20'}
colors_s = [RISK_COLORS.get(r,'#757575') for r in sectors['risk_level']]

fig, ax = plt.subplots(figsize=(9, 6))

for risk, grp in sectors.groupby('risk_level'):
    ax.scatter(grp['deforested_pct'], grp['hansen_pct'],
               color=RISK_COLORS.get(risk,'grey'),
               alpha=0.55, s=20 + grp['n_pixels']*0.15,
               label=f'{risk} ({len(grp)} sectors)', zorder=3)

# Regression line
from numpy.polynomial import polynomial as P
x_vals = sectors['deforested_pct'].values
y_vals = sectors['hansen_pct'].values
mask   = ~(np.isnan(x_vals) | np.isnan(y_vals))
c      = np.polyfit(x_vals[mask], y_vals[mask], 1)
x_line = np.linspace(0, 100, 100)
ax.plot(x_line, np.polyval(c, x_line), color='black', linewidth=1.5,
        linestyle='--', alpha=0.6, label=f'OLS fit')

pearson_r = hb['pearson_r']
spearman  = hb['spearman_rho']
ax.text(0.98, 0.05,
        f'Pearson r = {pearson_r:.3f}\nSpearman ρ = {spearman:.3f}\nn = {len(sectors)} sectors',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='#BDBDBD'))

ax.set_xlabel('TreeSight: % sector pixels predicted deforested', fontsize=10)
ax.set_ylabel('Hansen GFC: % sector pixels with recorded loss', fontsize=10)
ax.set_title('External validation — TreeSight vs Hansen GFC at sector level\n(independent ground truth, 365 Rwanda sectors)',
             fontsize=11, fontweight='bold')
ax.legend(frameon=False, fontsize=9, loc='upper left')
ax.grid(alpha=0.25)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(f'{METRICS}/nat_chart10_hansen_validation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Pearson r = {pearson_r:.3f}, Spearman ρ = {spearman:.3f}')
print('Note: moderate correlation expected — Hansen is pixel-level loss, we predict parcel risk from overlapping features.')
print('Cite as external triangulation, not a perfect benchmark.')

## Summary — all headline numbers in one place
For quick reference when writing the results chapter.

In [ ]:
spatial = nc['national_spatial_block_cv']

summary = {
    'Dataset size (national)':           f"{len(nat):,} pixels, 5 provinces",
    'Baseline target F1':                '0.71',
    'Random CV F1 (model D, 5-fold)':    f"{nc['four_experiments_5foldcv_200trees'][-1]['national_f1']:.3f}",
    'Spatial block CV F1 ← CITE THIS':  f"{spatial['f1']:.3f} ± {spatial['f1_std']:.3f}",
    'Spatial CV per-fold range':         f"{min(spatial['per_fold']):.3f} – {max(spatial['per_fold']):.3f}",
    'Feature importance: optical':       f"{na['by_source']['optical']:.1%}",
    'Feature importance: radar':         f"{na['by_source']['radar']:.1%}",
    'Feature importance: terrain':       f"{na['by_source']['terrain']:.1%}",
    'RQ1 top feature':                   na['feature_importance'][0]['feature'],
    'RQ2 recall @ 0.09-0.18 ha':         f"{rq2['by_patch_size'][0]['recall']:.3f}",
    'RQ2 recall @ >1.8 ha':              f"{rq2['by_patch_size'][-1]['recall']:.3f}",
    'Hansen validation Pearson r':       f"{hb['pearson_r']:.3f}",
    'Hansen validation Spearman ρ':      f"{hb['spearman_rho']:.3f}",
    'Province with weakest F1':          f"West ({nc['leave_one_province_out']['West/Iburengerazuba']['f1']:.3f})",
    'Province with strongest F1':        f"Kigali ({nc['leave_one_province_out']['Kigali City/Umujyi wa Kigali']['f1']:.3f})",
}

print('=' * 62)
print('TREESIGHT / UMURINZI — NATIONAL MODEL HEADLINE NUMBERS')
print('=' * 62)
for k, v in summary.items():
    marker = '  ★' if 'CITE' in k else ''
    print(f'{k:<42} {v}{marker}')
print('=' * 62)
print('\nCharts saved to results/metrics/nat_chart[1-10]_*.png')